In [1]:
import os
from pathlib import Path
import jwst
print(jwst.__version__)
from jwst import datamodels
from jwst.datamodels import dqflags

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
# import natural units:
import natural_units as nu
# multi-core/thread:
import concurrent.futures
import re


1.19.2


In [2]:
# 全局变量，一波定义完
number_of_electron_bins = 30
# bin 的范围。用 range(dn_min, dn_max+1) to include dn_max
dn_min = -200
dn_max = 400

# define the mass and cross section grid we are working with.
log_m_min  = -3
log_m_max  = 1
n_m    = 33   # From 1e-3  to 10 GeV
#log cs shift from the balloon line
log_cs_min = -6
log_cs_max = -3
n_cs   = 49
m_grid  = np.logspace(log_m_min, log_m_max, n_m)
cs_grid = np.logspace(log_cs_min, log_cs_max, n_cs)
center_line = np.array([2.15504637e-23, 1.88334907e-23, 1.53030461e-23, 1.32738030e-23,
       1.26359147e-23, 1.37889395e-23, 1.61669130e-23, 1.95514385e-23,
       2.40008514e-23, 3.14137096e-23, 4.03532201e-23, 5.19960700e-23,
       6.95747264e-23, 9.69011074e-23, 1.40957345e-22, 2.05768490e-22,
       2.84518575e-22, 3.82210762e-22, 5.17381239e-22, 7.29305911e-22,
       1.08351297e-21, 1.55439713e-21, 2.15203017e-21, 3.00211451e-21,
       4.12016417e-21, 5.67974728e-21, 7.78952222e-21, 1.06757849e-20,
       1.45615810e-20, 1.99263396e-20, 2.70914228e-20, 3.67600191e-20,
       4.94000000e-20])

# fraction rescale for 0.1% and 0.05%
frac = '1'
frac_rescale = float(frac)

In [3]:
#样本数量
sample_size = int(1e8)
# Generate pixel_value_raw by Poisson distribution:
def generate_raw_value(binned_signals):
    raw_value = np.zeros(sample_size)
    for i in range(len(binned_signals)):
        lambda_param = binned_signals[i]
        poisson_samples = np.random.poisson(lambda_param, sample_size)
        raw_value += (i+1) * poisson_samples
    return raw_value

binned_signal_path = Path('../data/binned_signal_halo_w_lindhard/')

def process_file(filename):
    file_stem = filename.stem
    # 使用 re.search() 匹配并提取
    match = re.search(r'(\d+)$', file_stem)
    m_index = int(match.group(1))  # 第一个括号组匹配 mDM 的值

    for cs_index in range(n_cs):
        if Path('./results_w_lin/DM_binned_halo_'+ frac +'/'+ str(m_index) + '_' + str(cs_index) +'.txt').exists():
            continue
        binned_signals = np.loadtxt(filename) * frac_rescale * cs_grid[cs_index]
        dm_sample = generate_raw_value(binned_signals)
        dm_poisson_counts, bin_edges = np.histogram(dm_sample, bins=range(dn_min, dn_max+1))
        dm_poisson_counts = dm_poisson_counts / sample_size   # to get PDF
        np.savetxt('./results_w_lin/DM_binned_halo_'+ frac +'/'+ str(m_index) + '_' + str(cs_index) +'.txt', dm_poisson_counts)
        print(file_stem[15:] + '  completed,     index' + str(m_index) + '_' + str(cs_index))

    return 0

binned_signal_files = [file for file in binned_signal_path.iterdir()]

In [4]:
# 使用 ProcessPoolExecutor 并行处理文件，限制最大进程数为 10
with concurrent.futures.ProcessPoolExecutor(max_workers=8) as executor:
    # 获取文件夹中所有的文件路径
    file_paths = [file for file in binned_signal_path.iterdir() if file.is_file()]
    
    # 提交文件处理任务到进程池
    futures = {executor.submit(process_file, file): file for file in file_paths}
    
    # 逐个处理完成的任务
    for future in concurrent.futures.as_completed(futures):
        file = futures[future]
        try:
            result = future.result()  # 获取任务的返回值
            print(f"Finished processing {result}")
        except Exception as exc:
            print(f"Error processing {file}: {exc}")

Halo_3  completed,     index3_28
Halo_5  completed,     index5_26
Halo_28  completed,     index28_25
Halo_22  completed,     index22_26
Halo_6  completed,     index6_25
Halo_21  completed,     index21_26
Halo_16  completed,     index16_26
Halo_29  completed,     index29_25
Halo_3  completed,     index3_29
Halo_28  completed,     index28_26
Halo_5  completed,     index5_27
Halo_22  completed,     index22_27
Halo_21  completed,     index21_27
Halo_6  completed,     index6_26
Halo_29  completed,     index29_26
Halo_16  completed,     index16_27
Halo_3  completed,     index3_30
Halo_28  completed,     index28_27
Halo_22  completed,     index22_28
Halo_5  completed,     index5_28
Halo_21  completed,     index21_28
Halo_6  completed,     index6_27
Halo_29  completed,     index29_27
Halo_16  completed,     index16_28
Halo_3  completed,     index3_31
Halo_28  completed,     index28_28
Halo_21  completed,     index21_29
Halo_22  completed,     index22_29
Halo_6  completed,     index6_28
Halo_5 

In [5]:
signal_grid = np.zeros((n_cs,n_m))
for m_index in range(n_m):
    filename = '../data/binned_signal_halo_w_lindhard/binned_signals_Halo_' + str(m_index) + '.txt'
    for cs_index in range(n_cs):
        binned_signals = np.loadtxt(filename) * frac_rescale * cs_grid[cs_index]
        for q in range(len(binned_signals)):
            signal_grid[n_cs -1 - cs_index, m_index] += (q+1) * binned_signals[q]

In [6]:
np.savetxt('../SHIELDING_RESULT/signal_grid_halo.txt',signal_grid, fmt='%.3e')